In [13]:
import os

os.chdir("../scripts")

from data_processing import poquad, processing
from t5 import load_t5

from Levenshtein import distance

In [14]:
train_df, valid_df = poquad.load_poquad_manually_downloaded("../data/poquad-manually-processed")

In [15]:
train_input = poquad.dataset_into_str_input(train_df)
valid_input = poquad.dataset_into_str_input(valid_df)

In [16]:
tokenizer_og, model_og = load_t5.load_plt5("../models/plt5-original-small")
tokenizer, model = load_t5.load_plt5("../models/plt5-small-2epochs")

In [27]:
sample = valid_input.iloc[7058:7059]
# sample = train_input.iloc[0:1]

distances = []

for i in range(1):
    output_og = tokenizer_og(sample["input_text"].iloc[i], return_tensors="pt").input_ids
    output = tokenizer(sample["input_text"].iloc[i], return_tensors="pt").input_ids

    gen_og = model_og.generate(output_og, max_length=128, num_beams=4, num_return_sequences=1, no_repeat_ngram_size=2, early_stopping=True)
    gen = model.generate(output, max_length=128, num_beams=4, num_return_sequences=1, no_repeat_ngram_size=2, early_stopping=True)

    gen_og_str = tokenizer_og.batch_decode(gen_og, skip_special_tokens=True)
    gen_str = tokenizer.batch_decode(gen, skip_special_tokens=True)

    dist_og = distance(sample["target_text"].iloc[i], gen_og_str[0])
    dist = distance(sample["target_text"].iloc[i], gen_str[0])

    distances.append((dist_og, dist))


In [24]:
valid_input

,input_text,target_text
0,kontekst: Miszna Pisma rabiniczne – w tym Mis...,odpowiedź: kompilację poglądów różnych rabinów...
1,kontekst: Miszna Pisma rabiniczne – w tym Mis...,odpowiedź: dwóch
2,kontekst: Miszna Pisma rabiniczne – w tym Mis...,"odpowiedź: pisanej, a drugą część w formie ustnej"
3,kontekst: Miszna Pisma rabiniczne – w tym Mis...,odpowiedź: ustna
4,kontekst: Miszna Pisma rabiniczne – w tym Mis...,odpowiedź: [BRAK_ODPOWIEDZI]
...,...,...
7055,kontekst: Bundeswehra Reforma sił zbrojnych N...,odpowiedź: 185 tys. żołnierzy i 40 tys. rezerw...
7056,kontekst: Bundeswehra Reforma sił zbrojnych N...,odpowiedź: istnieje jednak stały problem z doj...
7057,kontekst: Bundeswehra Reforma sił zbrojnych N...,odpowiedź: przewlekłych problemów ze sprzętem
7058,kontekst: Bundeswehra Reforma sił zbrojnych N...,odpowiedź: [BRAK_ODPOWIEDZI]


In [23]:
train_input.iloc[:20]

,input_text,target_text
0,kontekst: Konfederacja polsko-czechosłowacka ...,odpowiedź: wymianą listów Ripka – Stroński
1,kontekst: Konfederacja polsko-czechosłowacka ...,odpowiedź: E. Beneša i J. Masaryka z jednej a ...
2,kontekst: Konfederacja polsko-czechosłowacka ...,odpowiedź: podpisanie układu konfederacyjnego
3,kontekst: Konfederacja polsko-czechosłowacka ...,odpowiedź: E. Beneš
4,kontekst: Konfederacja polsko-czechosłowacka ...,odpowiedź: 20 listopada 1942
5,kontekst: Pomilio PD/PE Samoloty Pomilio PD w...,odpowiedź: Pomilio PD
6,kontekst: Pomilio PD/PE Samoloty Pomilio PD w...,odpowiedź: dwóch sekcji początkowo przydzielon...
7,kontekst: Pomilio PD/PE Samoloty Pomilio PD w...,odpowiedź: [BRAK_ODPOWIEDZI]
8,kontekst: Pomilio PD/PE Samoloty Pomilio PD w...,odpowiedź: PE
9,kontekst: Pomilio PD/PE Samoloty Pomilio PD w...,odpowiedź: [BRAK_ODPOWIEDZI]


In [21]:
sum([dist[0] > dist[1] for dist in distances])/len(distances)

0.91

In [17]:
sample["input_text"].iloc[i]

'kontekst: Konfederacja polsko-czechosłowacka  Projekty konfederacji zaczęły się załamywać 5 sierpnia 1942. Ponownie wróciła kwestia monachijska, co uaktywniło się wymianą listów Ripka – Stroński. Natomiast 17 sierpnia 1942 doszło do spotkania E. Beneša i J. Masaryka z jednej a Wł. Sikorskiego i E. Raczyńskiego z drugiej strony. Polscy dyplomaci zaproponowali podpisanie układu konfederacyjnego. W następnym miesiącu, tj. 24 września, strona polska przesłała na ręce J. Masaryka projekt deklaracji o przyszłej konfederacji obu państw. Strona czechosłowacka projekt przyjęła, lecz już w listopadzie 1942 E. Beneš podważył ideę konfederacji. W zamian zaproponowano zawarcie układu sojuszniczego z Polską na 20 lat (formalnie nastąpiło to 20 listopada 1942).  pytanie: Co było powodem powrócenia konceptu porozumieniu monachijskiego?'

In [18]:
gen_og_str

['規.̋.誰.嗨氀愀 [z: ciekawostka:']

In [19]:
gen_str

['規.̋.誰.嗨氀愀 [z: ciekawostka:']

In [20]:
tokenizer_og.batch_decode(gen_og, skip_special_tokens=False)

['<pad> 規.̋.誰.嗨氀愀 [z: ciekawostka:</s>']

In [21]:
sample = train_input.sample(100)

distances = []

for i in range(100):
    output_og = tokenizer_og(sample["input_text"].iloc[i], return_tensors="pt").input_ids
    output = tokenizer(sample["input_text"].iloc[i], return_tensors="pt").input_ids

    gen_og = model_og.generate(output_og, max_length=128, num_beams=4, num_return_sequences=1, no_repeat_ngram_size=2, early_stopping=True)
    gen = model.generate(output, max_length=128, num_beams=4, num_return_sequences=1, no_repeat_ngram_size=2, early_stopping=True)

    gen_og_str = tokenizer_og.batch_decode(gen_og, skip_special_tokens=True)
    gen_str = tokenizer.batch_decode(gen, skip_special_tokens=True)

    dist_og = distance(sample["target_text"].iloc[i], gen_og_str[0])
    dist = distance(sample["target_text"].iloc[i], gen_str[0])

    distances.append((dist_og, dist))


KeyboardInterrupt: 

In [ ]:
[dist for dist in distances if dist[0] != dist[1]]

[]